<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №11


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Customer в C#, который будет представлять информацию о 
клиентах или покупателях. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [7]:
public interface IContactable
{
    void SendEmail(string subject, string body);
    void Call(string message);
}

// БАЗОВЫЙ КЛАСС
public abstract class Customer : IContactable
{
    public int CustomerId { get; set; }
    public string Name { get; set; } 
    public string Email { get; set; }
    public string Phone { get; set; }    
    public bool IsActive { get; private set; } = true; 
    public DateTime CreatedAt { get; private set; } = DateTime.Now; 
    public string Address { get; set; } 

    public virtual string GetFullName() => Name;

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine(
            $"ID: {CustomerId}, Имя: {Name}, Email: {Email}, Телефон: {Phone}, Адрес: {Address}, Активен: {IsActive}, Создан: {CreatedAt:g}");
    }

    public void Activate()  => IsActive = true;
    public void Deactivate() => IsActive = false;
    public void ChangeName(string newName) => Name = newName;

    // ===== явная реализация интерфейса (скрыта за ссылкой IContactable) =====
    void IContactable.SendEmail(string subject, string body)
    {
        Console.WriteLine($"[Email → {Email}] Тема: {subject} | Сообщение: {body}");
    }

    void IContactable.Call(string message)
    {
        Console.WriteLine($"[Звонок → {Phone}] Сообщение: {message}");
    }
}


class VipCustomer : Customer
{
    public int LoyaltyPoints { get; set; }   
    public decimal DiscountRate { get; set; } 
    public DateTime VipStartDate { get; set; } 
    public string Tier { get; set; } = "Silver";

    public void AddPoints(int pts) => LoyaltyPoints += pts;

    public bool RedeemPoints(int pts)
    {
        if (LoyaltyPoints < pts) return false;
        LoyaltyPoints -= pts;
        return true;
    }

    public void UpgradeTier(string newTier) => Tier = newTier;

    public override void ViewProfile()
    {
        Console.WriteLine(
            $"[VIP] ID: {CustomerId}, Имя: {Name}, Email: {Email}, Телефон: {Phone}, Баллы: {LoyaltyPoints}, " +
            $"Скидка: {DiscountRate}%, Tier: {Tier}, VIP с: {VipStartDate:d}");
    }
}

class RegularCustomer : Customer
{
    public DateTime RegistrationDate { get; set; }
    public DateTime LastEmailUpdate { get; private set; }
    public int OrdersCount { get; private set; }
    public DateTime? LastLoginAt { get; private set; }
    public string PreferredLanguage { get; set; } = "ru"; 

    public void PlaceOrder()
    {
        OrdersCount++;
        Console.WriteLine($"{GetFullName()} оформил заказ. Всего заказов: {OrdersCount}");
    }

    public void Login()
    {
        LastLoginAt = DateTime.Now;
        Console.WriteLine($"{GetFullName()} вошёл в систему: {LastLoginAt:g}");
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
    }

    public override void ViewProfile()
    {
        Console.WriteLine(
            $"[REG] ID: {CustomerId}, Имя: {Name}, Email: {Email}, Телефон: {Phone}, Заказы: {OrdersCount}, " +
            $"Регистрация: {RegistrationDate:d}, Последнее обновление email: {LastEmailUpdate:g}, " +
            $"Язык: {PreferredLanguage}, Последний вход: {(LastLoginAt?.ToString("g") ?? "-")}");
    }
}

class GroupCustomer : Customer
{
    public string GroupName { get; set; }
    public int MembersCount { get; private set; }
    public string Coordinator { get; private set; }
    public string Department { get; set; }

    public void AddMember(int count = 1) => MembersCount += count;
    public void RemoveMember(int count = 1) => MembersCount = Math.Max(0, MembersCount - count);
    public void AssignCoordinator(string name) => Coordinator = name;

    public override string GetFullName() => GroupName;

    public override void ViewProfile()
    {
        Console.WriteLine(
            $"[GROUP] ID: {CustomerId}, Группа: {GroupName}, Email: {Email}, Телефон: {Phone}, " +
            $"Участников: {MembersCount}, Координатор: {Coordinator ?? "-"}, Отдел: {Department}");
    }
}

// УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (DI)
class NotificationService
{
    private readonly IContactable _contact;

    // Внедряем зависимость по интерфейсу — сервис ничего не знает о конкретном типе клиента
    public NotificationService(IContactable contact)
    {
        _contact = contact;
    }

    public void SendWelcome()
    {
        _contact.SendEmail("Добро пожаловать!", "Рады видеть вас среди наших клиентов.");
    }

    public void SendPromo(string text)
    {
        _contact.Call($"Промо-предложение: {text}");
    }
}


Console.WriteLine("=== Демонстрация явного интерфейса и DI ===\n");

Customer vip = new VipCustomer
{
    CustomerId = 1,
    Name = "Никита",
    Email = "nikita@mail.com",
    Phone = "+7 999 000-11-22",
    Address = "Тюмень, ул. Примерная, 1",
    LoyaltyPoints = 120,
    DiscountRate = 15,
    VipStartDate = new DateTime(2023, 10, 1),
    // Tier по умолчанию Silver
};

Customer regular = new RegularCustomer
{
    CustomerId = 2,
    Name = "Егор",
    Email = "egor@mail.com",
    Phone = "+7 900 123-45-67",
    Address = "Тюмень, ул. Учебная, 5",
    RegistrationDate = new DateTime(2022, 5, 1),
    PreferredLanguage = "ru"
};

Customer group = new GroupCustomer
{
    CustomerId = 3,
    Email = "group@mail.com",
    Phone = "+7 999 888-77-66",
    Address = "Тюмень, ул. Командная, 10",
    GroupName = "Студент"
};

// Работа производных методов
((VipCustomer)vip).AddPoints(30);
((VipCustomer)vip).UpgradeTier("Gold");

((RegularCustomer)regular).Login();
((RegularCustomer)regular).PlaceOrder();
regular.UpdateEmail("egor.new@mail.com");

((GroupCustomer)group).AddMember(12);
((GroupCustomer)group).AssignCoordinator("Анна");
((GroupCustomer)group).Department = "Обучение";

// Полиморфный вывод профилей
Console.WriteLine();
vip.ViewProfile();
regular.ViewProfile();
group.ViewProfile();

// явная реализация интерфейса: вызываем через IContactable
Console.WriteLine("\n--- Уведомления через явную реализацию интерфейса IContactable ---");
var vipNotifier = new NotificationService((IContactable)vip);
vipNotifier.SendWelcome();
vipNotifier.SendPromo("Скидка 20% на выходных!");

var regNotifier = new NotificationService((IContactable)regular);
regNotifier.SendWelcome();

var groupNotifier = new NotificationService((IContactable)group);
groupNotifier.SendPromo("Новые условия групповой подписки.");

Console.WriteLine("\n=== Конец демонстрации ===");


=== Демонстрация явного интерфейса и DI ===

Егор вошёл в систему: 10/17/2025 8:35 PM
Егор оформил заказ. Всего заказов: 1

[VIP] ID: 1, Имя: Никита, Email: nikita@mail.com, Телефон: +7 999 000-11-22, Баллы: 150, Скидка: 15%, Tier: Gold, VIP с: 10/1/2023
[REG] ID: 2, Имя: Егор, Email: egor.new@mail.com, Телефон: +7 900 123-45-67, Заказы: 1, Регистрация: 5/1/2022, Последнее обновление email: 10/17/2025 8:35 PM, Язык: ru, Последний вход: 10/17/2025 8:35 PM
[GROUP] ID: 3, Группа: Студент, Email: group@mail.com, Телефон: +7 999 888-77-66, Участников: 12, Координатор: Анна, Отдел: Обучение

--- Уведомления через явную реализацию интерфейса IContactable ---
[Email → nikita@mail.com] Тема: Добро пожаловать! | Сообщение: Рады видеть вас среди наших клиентов.
[Звонок → +7 999 000-11-22] Сообщение: Промо-предложение: Скидка 20% на выходных!
[Email → egor.new@mail.com] Тема: Добро пожаловать! | Сообщение: Рады видеть вас среди наших клиентов.
[Звонок → +7 999 888-77-66] Сообщение: Промо-предложен